# Fine-tuning with QLoRA (5-fold CV, one fold per session)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DSPagan/llms-time-complexity/blob/main/notebooks/fine_tuning.ipynb)

Fine-tune `Llama 3.1 8B Instruct` (4-bit) with QLoRA under **5-fold cross-validation**. Reloading five models in a single session would exhaust a T4, so each fold runs in its **own session**: set `FOLD`, train on the other four folds, evaluate on the held-out one, and save the fold's result. A final cell aggregates the five into **mean ± std**.

In [ ]:
# Clone the repo to get the code (src/), the data, and the pinned requirements.
!git clone https://github.com/DSPagan/llms-time-complexity.git
%cd llms-time-complexity

In [ ]:
# Install the exact pinned versions so every experiment runs on the same stack.
# Installing "latest" instead lets Unsloth drift between runs (and 2026.7.2 fails to
# resolve the 4-bit model repo), which would make the results incomparable.
!pip install --no-cache-dir -r requirements-lock.txt

In [ ]:
import os, sys, json
import numpy as np

sys.path.insert(0, os.getcwd())

from unsloth import FastLanguageModel
from src.load_model import load_model
from src.train_model import train_model
from src.prompts import build_prompt
from src.prepare_data import load_clean, stratified_folds, write_jsonl
from src.evaluate import evaluate, plot_confusion_matrix

In [ ]:
max_seq_length = 2048
num_epochs = 2
K = 5
folds = stratified_folds(load_clean(), k=K, seed=42)
print("fold sizes:", [len(f) for f in folds])

def predict(model, tokenizer, src, max_new_tokens=16):
    messages = [{"role": "user", "content": build_prompt(src)}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    if inputs.shape[1] > max_seq_length:
        return None
    out = model.generate(
        input_ids=inputs, do_sample=False, max_new_tokens=max_new_tokens,
        use_cache=True, no_repeat_ngram_size=4,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True).split("assistant")[-1].strip()

## Run one fold

Set `FOLD` to 0, 1, 2, 3 or 4 and run this **once per fold, each in a fresh runtime** — that way only one model is ever loaded (a T4 can't hold five, and reloading in-process leaks memory). The result is written to `outputs/finetune_fold_{FOLD}.json`; download it before closing the session.

In [ ]:
FOLD = 0   # <-- set to 0..4 across sessions; one fold per fresh runtime

test = folds[FOLD]
train = [x for j in range(K) if j != FOLD for x in folds[j]]
write_jsonl(train, "fold_train.jsonl")

model, tokenizer = load_model(max_seq_length=max_seq_length)
model, _ = train_model("fold_train.jsonl", model, tokenizer,
                       num_epochs=num_epochs, max_seq_length=max_seq_length)

FastLanguageModel.for_inference(model)
raw = [predict(model, tokenizer, item["src"]) for item in test]
res = evaluate([item["complexity"] for item in test], raw)

os.makedirs("outputs", exist_ok=True)
with open(f"outputs/finetune_fold_{FOLD}.json", "w") as f:
    json.dump({"fold": FOLD, "accuracy": res["accuracy"], "macro_f1": res["macro_f1"],
               "confusion_matrix": res["confusion_matrix"].tolist()}, f)

print(f"Fold {FOLD}: accuracy={res['accuracy']:.4f}  macro_f1={res['macro_f1']:.4f}")
print(f"saved -> outputs/finetune_fold_{FOLD}.json  (download it before closing)")

## Aggregate (after all 5 folds)

Once you have all five `outputs/finetune_fold_*.json` (re-upload them into `outputs/`, or keep them in Drive), this pools them into **mean ± std** and the combined confusion matrix.

In [ ]:
import glob

runs = [json.load(open(fp)) for fp in sorted(glob.glob("outputs/finetune_fold_*.json"))]
print(f"found {len(runs)} folds: {sorted(r['fold'] for r in runs)}")

accs = [r["accuracy"] for r in runs]
f1s = [r["macro_f1"] for r in runs]
print(f"Fine-tuned (QLoRA), {len(runs)}-fold CV:")
print(f"  accuracy  {np.mean(accs):.3f} +/- {np.std(accs, ddof=1):.3f}")
print(f"  macro_f1  {np.mean(f1s):.3f} +/- {np.std(f1s, ddof=1):.3f}")

cm = np.sum([np.array(r["confusion_matrix"]) for r in runs], axis=0)
os.makedirs("figures", exist_ok=True)
plot_confusion_matrix(cm, title=f"Fine-tuned (QLoRA, {len(runs)}-fold CV)",
                      save_path="figures/CM_finetuned.png")